# Consumer: Text Inference

This notebook shows how an external consumer can:
1. Call `/embed/text` to map raw text into the **exact** feature space used during training
2. Inspect the feature vector (which tags are present, at what score)
3. Call `/predict/text` to get a model prediction

The guarantee: the `feature_names` returned by the API are read directly from
`feature_names.txt` saved at training time. The tag vocabulary used to prompt
the LLM is read from `tag_vocabulary.json` — no new features can enter the
vector at inference time.

**Prerequisites:** The inference API must be running. Set `INFERENCE_URL` below.

## 1. Configuration

In [ ]:
import os

# When running via docker-compose, the service is on host port 8890.
# When deployed standalone on Kubeflow, use the cluster-internal URL.
INFERENCE_URL = os.getenv("INFERENCE_API_URL", "http://localhost:8890")

print(f"Inference API: {INFERENCE_URL}")

## 2. Health check

In [ ]:
import requests

resp = requests.get(f"{INFERENCE_URL}/health")
resp.raise_for_status()
health = resp.json()

print(f"Status         : {health['status']}")
print(f"Extractor ready: {health['extractor_ready']}")
print(f"Model ready    : {health['model_ready']}")
print(f"Data type      : {health['data_type']}")
print(f"N features     : {health['n_features']}")

## 3. Inspect the training feature space

The `/features` endpoint returns the ordered list of feature names exactly as
they appear in `feature_names.txt`. Every embedding vector from `/embed/text`
maps to this list in the same order.

In [ ]:
resp = requests.get(f"{INFERENCE_URL}/features")
resp.raise_for_status()
features_meta = resp.json()

print(f"Data type : {features_meta['data_type']}")
print(f"N features: {features_meta['n_features']}")
print()
print("Feature names (training vocabulary):")
for i, name in enumerate(features_meta['feature_names']):
    print(f"  [{i:2d}] {name}")

## 4. Embed a text sample

Replace `TEXT_SAMPLE` with any text from the same domain as the training data.
The API will score each training-vocabulary tag as present (1) or absent (0).

In [ ]:
TEXT_SAMPLE = "WINNER!! You have been selected for a free prize. Call now to claim!"

resp = requests.post(
    f"{INFERENCE_URL}/embed/text",
    json={"text": TEXT_SAMPLE},
)
resp.raise_for_status()
embed = resp.json()

print(f"N features returned : {embed['n_features']}")
print(f"Data type           : {embed['data_type']}")
print()
print("Feature vector:")
for name, value in zip(embed['feature_names'], embed['features']):
    bar = '█' * int(value * 20)
    print(f"  {name:<30s} {value:+.4f}  {bar}")

## 5. Verify alignment with training features

The cell below confirms that the feature names returned by `/embed/text` are
identical to those from `/features` — no surprise columns, no reordering.

In [ ]:
assert embed['feature_names'] == features_meta['feature_names'], (
    "Feature name mismatch between /embed/text and /features — this should never happen."
)
print("✓ Feature names are identical to the training feature space.")

## 6. Visualise the feature vector

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names  = embed['feature_names']
values = embed['features']

fig, ax = plt.subplots(figsize=(10, max(4, len(names) * 0.35)))
colors = ['steelblue' if v > 0 else 'lightgrey' for v in values]
ax.barh(names, values, color=colors)
ax.set_xlabel('Feature value (scaler-transformed)')
ax.set_title('Text embedding — feature vector')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Run model prediction

In [ ]:
resp = requests.post(
    f"{INFERENCE_URL}/predict/text",
    json={"text": TEXT_SAMPLE},
)
resp.raise_for_status()
pred = resp.json()

print(f"Score : {pred['score']:.4f}")
print(f"Label : {pred['label']}  ({'positive' if pred['label'] == 1 else 'negative'} class)")

## 8. Batch embedding

Call `/embed/text` once per sample. All responses share the same `feature_names`
list and can be stacked into a matrix for downstream processing.

In [ ]:
import pandas as pd

texts = [
    "WINNER!! You have been selected for a free prize. Call now to claim!",
    "Hey, are you coming to the meeting tomorrow at 10am?",
    "Urgent! Your account has been compromised. Click here to verify.",
]

rows = []
for text in texts:
    r = requests.post(f"{INFERENCE_URL}/embed/text", json={"text": text})
    r.raise_for_status()
    rows.append(r.json()['features'])

feature_names = requests.get(f"{INFERENCE_URL}/features").json()['feature_names']
df = pd.DataFrame(rows, columns=feature_names)
df.index = [f"sample_{i}" for i in range(len(texts))]
df